# ChartQAPro — End-to-End Pipeline Execution

Run the **complete agentic loop** from dataset loading through MEP generation,
evaluation, failure taxonomy, and summary — all from one notebook.

**How to use:** Edit the parameters in **Cell 1 (Configuration)**, then *Run All*.

| Section | What it does |
|---|---|
| 1 — Configuration | All tunable parameters in one place |
| 2 — Environment | Check API keys, install path, imports |
| 2.5 — Langfuse health check | Verify Langfuse credentials are configured before running |
| 3 — Load dataset | Pull samples from HuggingFace |
| 4 — Instantiate agents | Build Planner, OCR, Vision, Verifier |
| 5 — Run pipeline | Generate MEPs (Plan → OCR → Vision → Verify) |
| 6 — Inspect MEP | Browse the first result |
| 7 — OCR insight | See what OCR extracted and injected |
| 8 — Ablation: no OCR | Re-run without OCR to compare |
| 9 — Pass 1: eval outputs | Rule-based accuracy + optional LLM judge |
| 10 — Pass 2: eval traces | Latency, tool calls, replayability |
| 11 — Pass 4: failure taxonomy | VLM-based diagnosis of wrong answers |
| 12 — Summarize | Aggregate to summary.csv |

> **Companion notebook:** `analysis.ipynb` visualises existing results with charts and dashboards.
> This notebook *generates* the results.

## 1 — Configuration

Change these values. Everything else runs automatically.

In [ ]:
import sys
import uuid
from pathlib import Path


# ── Tune these before running ────────────────────────────────────────────────
N_SAMPLES = 1  # samples to process (keep ≤10 for a first run)
CONFIG = "gemini_gemini"  # gemini_gemini | openai_openai | openai_gemini | gemini_openai
SPLIT = "test"
USE_OCR = True  # False = skip OcrReaderTool  (--no_ocr equivalent)
USE_VERIFIER = True  # False = skip VerifierAgent  (--no_verifier equivalent)
OCR_MODEL = None  # None = same model as vision; 'gemini-2.5-flash-lite' = cheaper OCR
USE_JUDGE = False  # True = LLM rubric judge (extra API calls per sample)
RUN_ABLATION = True  # True = also run without OCR after the main run
# ────────────────────────────────────────────────────────────────────

REPO_ROOT = Path(".").resolve()
RUN_ID = str(uuid.uuid4())
OUT_DIR = str(REPO_ROOT / "meps" / CONFIG / "chartqapro" / SPLIT)
IMAGE_DIR = str(REPO_ROOT / "data" / "chartqapro_images")
OUT_LABEL = f"{CONFIG}_n{N_SAMPLES}"

ocr_status = "enabled" if USE_OCR else "disabled"
ver_status = "enabled" if USE_VERIFIER else "disabled"
print(f"Config     : {CONFIG}")
print(f"Samples    : {N_SAMPLES}  split={SPLIT}")
print(f"OCR        : {ocr_status}")
print(f"Verifier   : {ver_status}")
print(f"LLM judge  : {'enabled' if USE_JUDGE else 'disabled (--no_judge)'}")
print(f"Run ID     : {RUN_ID}")
print(f"Output dir : {OUT_DIR}")

## 2 — Environment

Verify API keys and import all modules. Fix any errors here before proceeding.

In [ ]:
import json
import os


# Make sure src/ is on the path
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dotenv import load_dotenv  # noqa: E402


load_dotenv(REPO_ROOT / ".env")

# API key check
missing = []
for var, needed_for in [("OPENAI_API_KEY", "openai"), ("GEMINI_API_KEY", "gemini")]:
    val = os.environ.get(var, "")
    needed = needed_for in CONFIG
    if val and not val.startswith("your_"):
        print(f"  ok  {var}  ({val[:3]}...)")
    elif needed:
        print(f"  MISSING  {var}  <- required for {CONFIG}")
        missing.append(var)
    else:
        print(f"  skip  {var}  (not needed for {CONFIG})")

if missing:
    raise EnvironmentError(f"Set {missing} in .env and re-run this cell.")

# Imports
import pandas as pd  # noqa: E402
from agentic_chartqapro_eval.agents.planner_agent import PlannerAgent  # noqa: E402
from agentic_chartqapro_eval.agents.verifier_agent import VerifierAgent  # noqa: E402
from agentic_chartqapro_eval.agents.vision_agent import VisionAgent  # noqa: E402
from agentic_chartqapro_eval.datasets.chartqapro_loader import (  # noqa: E402
    load_chartqapro,
)
from agentic_chartqapro_eval.eval.error_taxonomy import classify_failure  # noqa: E402
from agentic_chartqapro_eval.eval.eval_outputs import evaluate_mep  # noqa: E402
from agentic_chartqapro_eval.eval.eval_topk import evaluate_topk  # noqa: E402
from agentic_chartqapro_eval.eval.eval_traces import evaluate_trace  # noqa: E402
from agentic_chartqapro_eval.eval.summarize import summarize, write_csv  # noqa: E402
from agentic_chartqapro_eval.langfuse_integration.client import get_client  # noqa: E402
from agentic_chartqapro_eval.mep.writer import iter_meps  # noqa: E402
from agentic_chartqapro_eval.runner.run_generate_meps import (  # noqa: E402
    BACKEND_CONFIGS,
    process_sample,
)
from agentic_chartqapro_eval.tools.ocr_reader_tool import OcrReaderTool  # noqa: E402
from IPython.display import Image, display  # noqa: E402, A004


print("\nAll imports OK")

## 2.5 — Langfuse Health Check

Verifies that Langfuse credentials are configured before the pipeline runs.

| Check | What it tests |
|---|---|
| Env vars present | `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are set in `.env` |
| Client init | `Langfuse()` initialises without error |

If the keys are absent the cell prints a skip notice and continues — Langfuse is optional.
The pipeline produces identical MEPs with or without it; tracing is purely additive.

In [ ]:
from agentic_chartqapro_eval.langfuse_integration.client import reset_client


# Force re-initialisation so re-running this cell picks up any .env changes
reset_client()

lf_public = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
lf_secret = os.environ.get("LANGFUSE_SECRET_KEY", "")

if not lf_public or not lf_secret:
    print("[skip] LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY are not set.")
    print("       Langfuse tracing is disabled. Pipeline will run fine without it.")
    print()
    print("To enable Langfuse tracing, add to .env:")
    print("  LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("  LANGFUSE_SECRET_KEY=sk-lf-...")
    print("  # LANGFUSE_HOST=https://cloud.langfuse.com  (default; change for self-hosted)")
else:
    results = {}

    # -- Check 1: Env vars present --
    results["env"] = ("ok", f"pk={lf_public[:3]}...")

    # -- Check 2: Client initialises --
    try:
        _lf_hc = get_client()
        if _lf_hc is not None:
            results["client"] = ("ok", "Langfuse() ready")
        else:
            results["client"] = ("fail", "get_client() returned None")
    except Exception as e:
        results["client"] = ("fail", str(e))

    # -- Report --
    labels = [
        ("env", "Env vars present"),
        ("client", "Client init     "),
    ]
    all_ok = True
    for key, label in labels:
        status, detail = results.get(key, ("skip", ""))
        marker = "✓ OK  " if status == "ok" else ("⊘ skip" if status == "skip" else "✗ FAIL")
        if status not in ("ok", "skip"):
            all_ok = False
        print(f"  {marker}  {label}  {detail}")

    print()
    if all_ok:
        lf_host = os.environ.get("LANGFUSE_HOST") or os.environ.get("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
        print("✓ Langfuse is configured.")
        print(f"Host      : {lf_host}")
        print("Traces and scores will be recorded automatically during the pipeline run.")
    else:
        print("⚠ WARNING: Langfuse client failed to initialise.")
        print("The pipeline will still run; tracing will be skipped.")

## 3 — Load Dataset

Downloads ChartQAPro from HuggingFace (cached after first run) and saves chart images locally.

In [ ]:
from collections import Counter


Path(IMAGE_DIR).mkdir(parents=True, exist_ok=True)

samples = load_chartqapro(split=SPLIT, n=N_SAMPLES, image_dir=IMAGE_DIR)
print(f"Loaded {len(samples)} samples from ChartQAPro ({SPLIT} split)\n")

qt_counts = Counter(s.question_type.value for s in samples)
print("Question type breakdown:")
for qt, n in sorted(qt_counts.items()):
    print(f"  {qt:<22} {n}")

# Peek at first sample
s0 = samples[0]
print(f"\nFirst sample: {s0.sample_id}")
print(f"  Q: {s0.question}")
print(f"  A: {s0.expected_output}")
print(f"  Type: {s0.question_type.value}")

## 4 — Instantiate Agents

Build the four components of the pipeline. Agents are created once and reused across all samples.
CrewAI creates a fresh `Crew` per `run()` call, so this is thread-safe for parallel workers.

In [ ]:
config = dict(BACKEND_CONFIGS[CONFIG])
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# PlannerAgent: text-only LLM, no image access
planner = PlannerAgent(
    backend=config["planner_backend"],
    model=config["planner_model"],
)
print(f"PlannerAgent   : {config['planner_backend']} / {config['planner_model']}")

# VisionAgent: multimodal LLM + VisionQATool
vision_agent = VisionAgent(
    agent_backend=config["planner_backend"],
    agent_model=config["planner_model"],
    vision_backend=config["vision_backend"],
    vision_model=config["vision_model"],
)
print(f"VisionAgent    : {config['vision_backend']} / {config['vision_model']}")

# VerifierAgent: optional Pass 2.5 critic
verifier = None
if USE_VERIFIER:
    verifier = VerifierAgent(backend=config["vision_backend"], model=config["vision_model"])
    print(f"VerifierAgent  : {config['vision_backend']} / {config['vision_model']}")
else:
    print("VerifierAgent  : disabled (USE_VERIFIER=False)")

# OcrReaderTool: optional perception pre-pass
ocr = None
if USE_OCR:
    ocr_model = OCR_MODEL or config["vision_model"]
    ocr = OcrReaderTool(backend=config["vision_backend"], model=ocr_model)
    print(f"OcrReaderTool  : {config['vision_backend']} / {ocr_model}")
else:
    print("OcrReaderTool  : disabled (USE_OCR=False)")

# Langfuse observability (no-op if keys not set)
lf_client = get_client()
lf_status = "enabled" if lf_client else "not configured"
print(f"Langfuse       : {lf_status}")

## 5 — Run the Pipeline

For each sample: **Plan → [OCR] → Vision → [Verify] → write MEP**.

Each call to `process_sample()` is fully self-contained: it creates a fresh CrewAI `Crew`,
runs the agent chain, and serialises the result to `OUT_DIR/<sample_id>.json`.

To parallelise, increase `N_WORKERS` below and run again.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


N_WORKERS = 1  # set >1 to parallelise (increase N_SAMPLES too)

mep_paths = []
print(f"Running {len(samples)} samples  workers={N_WORKERS}  -> {OUT_DIR}\n")


def _run_one(sample):
    return process_sample(
        sample,
        planner,
        vision_agent,
        config,
        RUN_ID,
        OUT_DIR,
        lf_client=lf_client,
        verifier_agent=verifier,
        ocr_tool=ocr,
    )


if N_WORKERS <= 1:
    for i, sample in enumerate(samples, 1):
        print(f"[{i}/{len(samples)}] {sample.sample_id} ...", end=" ", flush=True)
        try:
            path = _run_one(sample)
            mep_paths.append(path)
            print("OK")
        except Exception as exc:
            print(f"ERROR: {exc}")
else:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_run_one, s): s for s in samples}
        for done, fut in enumerate(as_completed(futures), 1):
            s = futures[fut]
            try:
                path = fut.result()
                mep_paths.append(path)
                print(f"[{done}/{len(samples)}] {s.sample_id} -> OK")
            except Exception as exc:
                print(f"[{done}/{len(samples)}] {s.sample_id} ERROR: {exc}")

print(f"\nDone -- {len(mep_paths)}/{len(samples)} MEPs written to {OUT_DIR}")

## 6 — Inspect First MEP

MEPs are self-contained JSON files. Every field you see here is what the agent actually
produced — no post-processing. The `lf_trace_id` links this MEP back to the live trace
in the Langfuse dashboard if Langfuse is configured.

In [ ]:
if not mep_paths:
    print("No MEPs written -- check errors above.")
else:
    mep = json.loads(Path(sorted(mep_paths)[0]).read_text())
    s = mep["sample"]
    pl = mep.get("plan", {}).get("parsed", {})
    vi = mep.get("vision", {}).get("parsed", {})
    vr = (mep.get("verifier") or {}).get("parsed", {})
    ts = mep.get("timestamps", {})

    print("=" * 64)
    print(f"Sample   : {s['sample_id']}")
    print(f"Type     : {s['question_type']}")
    print(f"Question : {s['question']}")
    print(f"Expected : {s['expected_output']}")
    print()
    print("Planner steps:")
    for j, step in enumerate(pl.get("steps", []), 1):
        print(f"  {j}. {step}")
    print()
    print(f"Vision answer : {vi.get('answer', '--')}")
    print(f"Explanation   : {vi.get('explanation', '--')[:280]}")
    if vr:
        print()
        print(f"Verifier verdict  : {vr.get('verdict', '--')}")
        print(f"Verifier answer   : {vr.get('answer', '--')}")
        print(f"Verifier reasoning: {vr.get('reasoning', '--')}")
    print()
    print("Timestamps (ms):")
    for k in ["planner_ms", "ocr_ms", "vision_ms", "verifier_ms"]:
        print(f"  {k:<16} {ts.get(k, 0):.0f}")
    if mep.get("lf_trace_id"):
        print(f"Langfuse trace ID: {mep['lf_trace_id']}")
    print("=" * 64)

    img_path = s.get("image_ref", {}).get("path", "")
    if img_path and Path(img_path).exists():
        display(Image(filename=img_path, width=640))
    else:
        print(f"(image not found at: {img_path})")

## 7 — OCR Insight: What Was Injected into the Vision Prompt?

The OcrReaderTool makes a single VLM call focused purely on text transcription.
Its structured JSON output is injected into the VisionAgent's prompt as `{ocr_block}`.

This cell shows exactly what was transcribed and injected. When `USE_OCR=False`,
`{ocr_block}` renders as an empty string and the prompt is otherwise unchanged —
a clean example of conditional context injection without prompt branching.

In [ ]:
if not mep_paths:
    print("No MEPs to inspect.")
else:
    mep = json.loads(Path(sorted(mep_paths)[0]).read_text())
    ocr_data = mep.get("ocr")

    if ocr_data is None:
        print("OCR was disabled (ocr: null in MEP).")
        print("The {ocr_block} placeholder in vision.txt rendered as an empty string.")
        print("Re-run with USE_OCR=True to see OCR output.")
    else:
        parsed = ocr_data.get("parsed", {})
        ts = mep.get("timestamps", {})
        print(f"OCR elapsed     : {ts.get('ocr_ms', 0):.0f} ms")
        print(f"Parse error     : {ocr_data.get('parse_error', False)}")
        print()
        print("Structured OCR output (injected as {ocr_block} into vision.txt):")
        print(f"  chart_type  : {parsed.get('chart_type', '--')}")
        print(f"  title       : {parsed.get('title', '--')}")
        x = parsed.get("x_axis", {})
        print(f"  x_axis      : label={x.get('label', '--')}  ticks={x.get('ticks', [])}")
        y = parsed.get("y_axis", {})
        print(f"  y_axis      : label={y.get('label', '--')}  ticks={y.get('ticks', [])}")
        print(f"  legend      : {parsed.get('legend', [])}")
        print(f"  data_labels : {parsed.get('data_labels', [])}")
        print(f"  annotations : {parsed.get('annotations', [])}")
        print()
        # Show raw OCR text
        raw = ocr_data.get("raw_text", "")
        print("Raw OCR response (first 400 chars):")
        print(raw[:400])

## 8 — Ablation: Run Without OCR

Runs the same samples with `ocr_tool=None` and writes to a separate directory.
This lets you compare the two runs side-by-side in Pass 1 evaluation.

Set `RUN_ABLATION=False` in the config cell to skip this section.

In [ ]:
no_ocr_paths = []
OUT_DIR_NO_OCR = str(REPO_ROOT / "meps" / (CONFIG + "_no_ocr") / "chartqapro" / SPLIT)
RUN_ID_NO_OCR = str(uuid.uuid4())

if not RUN_ABLATION:
    print("Skipped (RUN_ABLATION=False).")
elif not USE_OCR:
    print("USE_OCR is already False -- the main run above IS the no-OCR baseline.")
    print("Set USE_OCR=True and re-run to get the with-OCR version for comparison.")
else:
    Path(OUT_DIR_NO_OCR).mkdir(parents=True, exist_ok=True)
    print(f"Running {len(samples)} samples WITHOUT OCR -> {OUT_DIR_NO_OCR}\n")
    for i, sample in enumerate(samples, 1):
        print(f"[{i}/{len(samples)}] {sample.sample_id} ...", end=" ", flush=True)
        try:
            path = process_sample(
                sample,
                planner,
                vision_agent,
                config,
                RUN_ID_NO_OCR,
                OUT_DIR_NO_OCR,
                lf_client=lf_client,
                verifier_agent=verifier,
                ocr_tool=None,  # <-- OCR disabled
            )
            no_ocr_paths.append(path)
            print("OK")
        except Exception as exc:
            print(f"ERROR: {exc}")
    print(f"\nDone -- {len(no_ocr_paths)} no-OCR MEPs written")
    print("\nOCR vs no-OCR: check ocr_ms in timestamps for the time cost:")
    if mep_paths:
        mep_with = json.loads(Path(sorted(mep_paths)[0]).read_text())
        mep_without = json.loads(Path(no_ocr_paths[0]).read_text())
        print(
            f"  With OCR    ocr_ms={mep_with.get('timestamps', {}).get('ocr_ms', 0):.0f}  vision_ms={mep_with.get('timestamps', {}).get('vision_ms', 0):.0f}"
        )
        print(
            f"  Without OCR ocr_ms={mep_without.get('timestamps', {}).get('ocr_ms', 0):.0f}  vision_ms={mep_without.get('timestamps', {}).get('vision_ms', 0):.0f}"
        )

## 9 — Pass 1: Evaluate Outputs

Rule-based accuracy scoring + optional LLM judge (5 rubric dimensions).

- `answer_accuracy`: exact-match with numeric tolerance; MCQ partial credit
- `verifier_verdict`: `confirmed` / `revised` / `skipped`
- `judge_*`: scored 0–1 by a text LLM (only when `USE_JUDGE=True`)

Results are written to `metrics_<label>.jsonl` in the repo root.

In [ ]:
metrics_path = str(REPO_ROOT / "output" / f"metrics_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
metrics_list = []

print(f"Evaluating MEPs in {OUT_DIR} ...")
with open(metrics_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        try:
            m = evaluate_mep(
                mep_dict,
                use_judge=USE_JUDGE,
                judge_backend=config.get("judge_backend", "gemini"),
                judge_model=config.get("vision_model", "gemini-2.5-flash-lite"),
            )
            f_out.write(json.dumps(m) + "\n")
            metrics_list.append(m)
        except Exception as exc:
            print(f"  eval error: {exc}")

print(f"Pass 1 complete -- {len(metrics_list)} rows -> {metrics_path}\n")

df = pd.DataFrame(metrics_list)
print(f"Overall accuracy : {df['answer_accuracy'].mean():.1%}  (n={len(df)})")
if "verifier_verdict" in df.columns and not df["verifier_verdict"].eq("skipped").all():
    revised = (df["verifier_verdict"] == "revised").sum()
    print(f"Verifier revised : {revised}/{len(df)} ({revised / len(df):.1%})")

# Accuracy by question type
print()
print("Accuracy by question type:")
for qt, grp in df.groupby("question_type"):
    print(f"  {qt:<22} {grp['answer_accuracy'].mean():.1%}  (n={len(grp)})")

In [ ]:
# Compare OCR vs no-OCR accuracy (only runs if ablation was done)
if no_ocr_paths:
    no_ocr_metrics = []
    for mep_dict in iter_meps(OUT_DIR_NO_OCR):
        import contextlib

        with contextlib.suppress(Exception):
            no_ocr_metrics.append(evaluate_mep(mep_dict, use_judge=False))
    df_no_ocr = pd.DataFrame(no_ocr_metrics)
    print("Accuracy comparison:")
    print(f"  WITH OCR    : {df['answer_accuracy'].mean():.1%}  (n={len(df)})")
    print(f"  WITHOUT OCR : {df_no_ocr['answer_accuracy'].mean():.1%}  (n={len(df_no_ocr)})")
    delta = df["answer_accuracy"].mean() - df_no_ocr["answer_accuracy"].mean()
    print(f"  Delta       : {delta:+.1%}  (positive = OCR helped)")
else:
    print("Ablation not run -- set RUN_ABLATION=True and re-run to compare OCR vs no-OCR.")

## 10 — Pass 2: Evaluate Traces

Trace-level metrics: latency per agent, tool call counts, and replayability score
(fraction of required MEP fields that are present and non-empty).

In [ ]:
trace_path = str(REPO_ROOT / "output" / f"trace_metrics_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
trace_rows = []

with open(trace_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        row = evaluate_trace(mep_dict)
        f_out.write(json.dumps(row) + "\n")
        trace_rows.append(row)

trace_df = pd.DataFrame(trace_rows)
print(f"Pass 2 complete -- {len(trace_df)} rows -> {trace_path}\n")

cols = [
    "latency_sec",
    "planner_latency_sec",
    "vision_latency_sec",
    "tool_call_count",
    "replayability",
    "error_count",
]
cols = [c for c in cols if c in trace_df.columns]
print(trace_df[cols].describe().round(3).to_string())

## 11 — Pass 4: Failure Taxonomy

For each **wrong** answer, a VLM is shown the chart image, the wrong answer,
the correct answer, and the agent's explanation — and classifies the primary failure mode.

Unlike a text-only judge, this pass can verify whether axis labels were actually ambiguous,
whether the cited data point appears in the image, etc.

Correct answers (`answer_accuracy == 1.0`) are skipped without making any API call.

In [ ]:
taxonomy_path = str(REPO_ROOT / "output" / f"taxonomy_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
taxonomy_rows = []

# Build accuracy lookup from Pass 1
acc_map = {r["sample_id"]: r["answer_accuracy"] for r in metrics_list}

vlm_backend = config["vision_backend"]
vlm_model = config["vision_model"]

print(f"Classifying failures using {vlm_backend} / {vlm_model} ...")
with open(taxonomy_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        sid = mep_dict.get("sample", {}).get("sample_id", "")
        acc = acc_map.get(sid, 0.0)
        try:
            result = classify_failure(
                mep_dict,
                answer_accuracy=acc,
                backend=vlm_backend,
                model=vlm_model,
            )
            row = {"sample_id": sid, "answer_accuracy": acc, **result}
            f_out.write(json.dumps(row) + "\n")
            taxonomy_rows.append(row)
            marker = "(skipped -- correct)" if acc == 1.0 else result.get("failure_type", "")
            print(f"  {sid}: {marker}")
        except Exception as exc:
            print(f"  {sid}: ERROR {exc}")

tax_df = pd.DataFrame(taxonomy_rows)
print(f"\nPass 4 complete -- {len(tax_df)} rows -> {taxonomy_path}\n")
print("Failure type breakdown:")
print(tax_df["failure_type"].value_counts().to_string())

## 12 — Top-K Evaluation (hit@1 / hit@2 / hit@3)

Re-queries the VLM for each MEP asking for the top-K most likely candidate answers.
Computes hit@k: whether the expected answer appears within the top-k candidates.

- **hit@1** ≈ standard accuracy with a fresh independent call
- **hit@2/3** measures how often the right answer is in the model's top candidates

In [ ]:
topk_path = str(REPO_ROOT / "output" / f"topk_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
topk_list = []

K = 3
topk_backend = config.get("vision_backend", "gemini")
topk_model = config.get("vision_model", "gemini-2.5-flash-lite")

print(f"Running top-{K} evaluation on MEPs in {OUT_DIR} ...")
print(f"  backend={topk_backend}  model={topk_model}\n")

with open(topk_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        try:
            result = evaluate_topk(
                mep_dict,
                k=K,
                backend=topk_backend,
                model=topk_model,
            )
            f_out.write(json.dumps(result) + "\n")
            topk_list.append(result)
            sid = result["sample_id"]
            cands = result["topk_candidates"]
            h1 = result.get("hit_at_1", 0)
            hk = result.get(f"hit_at_{K}", 0)
            print(f"  {sid}  candidates={cands}  hit@1={h1:.0f}  hit@{K}={hk:.0f}")
        except Exception as exc:
            print(f"  eval error: {exc}")

print(f"\nTop-{K} complete -- {len(topk_list)} rows -> {topk_path}\n")

if topk_list:
    orig_acc = sum(r["original_accuracy"] for r in topk_list) / len(topk_list)
    print(f"Original accuracy (from MEP)  : {orig_acc:.1%}")
    for ki in range(1, K + 1):
        key = f"hit_at_{ki}"
        val = sum(r.get(key, 0) for r in topk_list) / len(topk_list)
        print(f"hit@{ki}                         : {val:.1%}")

## 13 — Summarize

Aggregate all metrics into `summary_<label>.csv`, grouped by `config_name` and `question_type`.
To compare multiple configs, run cells 5–12 again with a different `CONFIG` value
then concatenate the `metrics_*.jsonl` files before summarising.

In [ ]:
summary_path = str(REPO_ROOT / "output" / f"summary_{OUT_LABEL}.csv")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)

rows = summarize(metrics_list)
write_csv(rows, summary_path)
print(f"Summary written -> {summary_path}  ({len(rows)} rows)\n")

summary_df = pd.DataFrame(rows)
rename = {
    "answer_accuracy_mean": "accuracy",
    "latency_sec_mean": "latency_s",
    "tool_call_count_mean": "tool_calls",
}
show_cols = [
    "config_name",
    "question_type",
    "count",
    "accuracy",
    "latency_s",
    "tool_calls",
]
show_cols = [c for c in show_cols if c in [rename.get(k, k) for k in summary_df.columns] + list(summary_df.columns)]

display(
    summary_df.rename(columns=rename)[
        [
            c
            for c in [
                "config_name",
                "question_type",
                "count",
                "accuracy",
                "latency_s",
                "tool_calls",
            ]
            if c in summary_df.rename(columns=rename).columns
        ]
    ]
    .fillna("")
    .style.format({"accuracy": "{:.1%}", "latency_s": "{:.1f}", "tool_calls": "{:.1f}"})
)

## What to Explore Next

- **`analysis.ipynb`** — visualise the results you just generated: accuracy charts, verifier
  revision analysis, failure taxonomy bar chart, judge score distributions, per-sample browser

- **Compare configs** — change `CONFIG` in Cell 1 to `gemini_gemini` or `openai_gemini`,
  re-run cells 5–12, then concatenate the two `metrics_*.jsonl` files and summarise together

- **Model overrides** — set `OCR_MODEL = 'gpt-4o-mini'` to run a cheaper model for OCR
  while keeping `gpt-4o` for the VisionAgent

- **Streamlit dashboard** — `streamlit run src/agentic_chartqapro_eval/eval/dashboard.py`

- **HTML report** — `uv run --env-file .env -m agentic_chartqapro_eval.eval.report --metrics metrics_<label>.jsonl --taxonomy taxonomy_<label>.jsonl --out report.html`